# Tối ưu hóa mô hình TabNet bằng thuật toán Particle Swarm Optimization cho dự đoán khách hàng rời bỏ trong thương mại điện tử

---


## Tổng quan nghiên cứu

Dự đoán khách hàng rời bỏ là một chủ đề được nghiên cứu rộng rãi trong thương mại điện tử và quản trị quan hệ khách hàng (CRM). Trong bối cảnh cạnh tranh ngày càng gay gắt, việc duy trì khách hàng hiện tại thường được xem là hiệu quả về chi phí và bền vững hơn so với việc liên tục thu hút khách hàng mới. Do đó, các mô hình dự đoán rời bỏ giữ vai trò quan trọng trong việc hỗ trợ doanh nghiệp chủ động nhận diện những khách hàng có nguy cơ rời đi, từ đó xây dựng và triển khai các biện pháp can thiệp phù hợp.

Bài toán dự đoán rời bỏ thường được xây dựng dưới dạng bài toán phân loại nhị phân, trong đó mục tiêu là ước lượng xác suất một khách hàng sẽ ngừng sử dụng dịch vụ trong một khoảng thời gian nhất định. Tuy nhiên, dữ liệu rời bỏ trong thương mại điện tử thường mang đặc trưng của dữ liệu dạng bảng, bao gồm nhiều kiểu biến khác nhau, tồn tại các mối quan hệ phi tuyến giữa các đặc trưng, đồng thời thường gặp tình trạng mất cân bằng lớp. Những đặc điểm này đặt ra yêu cầu cần lựa chọn mô hình phù hợp để khai thác hiệu quả cấu trúc dữ liệu và nâng cao độ chính xác dự đoán.

---

## Mục tiêu nghiên cứu

Nghiên cứu này tập trung vào các mục tiêu chính sau:

- Phân tích đặc điểm dữ liệu khách hàng và các yếu tố ảnh hưởng đến hành vi rời bỏ  
- Xây dựng mô hình dự đoán rời bỏ khách hàng dựa trên kiến trúc TabNet dành cho dữ liệu dạng bảng  
- Ứng dụng thuật toán Particle Swarm Optimization (PSO) để tối ưu siêu tham số của mô hình  
- So sánh hiệu quả của mô hình TabNet được tối ưu với các mô hình cơ sở như:
  - Logistic Regression  
  - Random Forest  
  - XGBoost  
- Đánh giá hiệu suất mô hình thông qua các chỉ số:
  - F1-score  
  - Recall  
  - ROC-AUC  
  - PR-AUC  
- Phân tích mức độ quan trọng của các biến nhằm xác định những yếu tố tác động mạnh đến khả năng khách hàng rời bỏ  

---

## Cấu trúc nghiên cứu

Nghiên cứu được triển khai theo bốn giai đoạn chính:

1. Chuẩn bị dữ liệu và thiết lập thực nghiệm  
2. Phân tích khám phá dữ liệu (EDA)  
3. Xây dựng mô hình dự đoán và tối ưu siêu tham số  
   - Các mô hình cơ sở  
   - TabNet với cấu hình mặc định  
   - TabNet kết hợp tối ưu bằng PSO  
4. Đánh giá mô hình, diễn giải kết quả và hàm ý quản trị  

---

## Phase 1: Notebook Setup & Reproducibility

1. Thiết lập notebook và đảm bảo khả năng tái lập kết quả  
2. Tổng quan dữ liệu  
   - Kích thước dữ liệu  
   - Xem trước dữ liệu  
   - Mô tả các biến  
   - Kiểu dữ liệu của từng cột  
   - Thống kê mô tả  
   - Giá trị thiếu  
   - Phân bố lớp  
3. Làm sạch dữ liệu ban đầu  
4. Chia dữ liệu phục vụ thực nghiệm  
5. Xử lý giá trị thiếu (chỉ fit trên tập huấn luyện)  
6. Mã hóa biến và chuẩn hóa dữ liệu  
7. Tóm tắt dữ liệu sau tiền xử lý  

In [1]:
import os
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings("ignore")

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def create_folders():
    os.makedirs("artifacts/data", exist_ok=True)
    os.makedirs("artifacts/results", exist_ok=True)
    os.makedirs("artifacts/models", exist_ok=True)
    os.makedirs("artifacts/figures", exist_ok=True)

def save_figure(fig, filename, folder="artifacts/figures", dpi=300):
    Path(folder).mkdir(parents=True, exist_ok=True)
    fig.savefig(Path(folder) / filename, dpi=dpi, bbox_inches="tight")

set_seed()
create_folders()

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

#### **1. Notebook Setup & Reproducibility**

In [2]:
warnings.filterwarnings('ignore')

#read file data from google drive
url_data = 'https://drive.google.com/file/d/19hAFPyZSII8ghztnqicmKxffWNhVSxBS/view?usp=sharing'
url_data = 'https://drive.google.com/uc?id=' + url_data.split('/')[-2]
raw_data = pd.read_csv(url_data)

#read file codebook from google drive
url_cols = 'https://drive.google.com/file/d/1dIWX67E2B4fEKM0sHU9iKneMhOufH6QF/view?usp=sharing'
url_cols = 'https://drive.google.com/uc?id=' + url_cols.split('/')[-2]
cols = pd.read_csv(url_cols, index_col=0)
raw_data

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,160
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,121
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5625,55626,0,10.0,Computer,1,30.0,Credit Card,Male,3.0,2,Laptop & Accessory,1,Married,6,0,18.0,1.0,2.0,4.0,151
5626,55627,0,13.0,Mobile Phone,1,13.0,Credit Card,Male,3.0,5,Fashion,5,Married,6,0,16.0,1.0,2.0,NaN,225
5627,55628,0,1.0,Mobile Phone,1,11.0,Debit Card,Male,3.0,2,Laptop & Accessory,4,Married,3,1,21.0,1.0,2.0,4.0,186
5628,55629,0,23.0,Computer,3,9.0,Credit Card,Male,4.0,5,Laptop & Accessory,4,Married,4,0,15.0,2.0,2.0,9.0,179


#### **2. Data overview** 

In [3]:
print('Shape of data:')
raw_data.shape

Shape of data:


(5630, 20)

In [4]:
print('Data preview:')
raw_data.head()

Data preview:


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,160
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,121
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,130


In [5]:
#column description
cols

,Variable,Discerption
Data,,
E Comm,CustomerID,Unique customer ID
E Comm,Churn,Churn Flag
E Comm,Tenure,Tenure of customer in organization
E Comm,PreferredLoginDevice,Preferred login device of customer
E Comm,CityTier,City tier
E Comm,WarehouseToHome,Distance in between warehouse to home of customer
E Comm,PreferredPaymentMode,Preferred payment method of customer
E Comm,Gender,Gender of customer
E Comm,HourSpendOnApp,Number of hours spend on mobile application or...


In [6]:
pd.set_option('display.max_rows', 80)
print('Data type per column:')
raw_data.dtypes

Data type per column:


CustomerID                       int64
Churn                            int64
Tenure                         float64
PreferredLoginDevice            object
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode            object
Gender                          object
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferedOrderCat                object
SatisfactionScore                int64
MaritalStatus                   object
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear    float64
CouponUsed                     float64
OrderCount                     float64
DaySinceLastOrder              float64
CashbackAmount                   int64
dtype: object

In [7]:
print("STATISTICAL SUMMARY")

print("Statistical summary (numerical variables):")
display(raw_data.drop(columns=["CustomerID"]).describe())

print("\nCategorical summary:")
display(raw_data.describe(include='object'))

STATISTICAL SUMMARY
Statistical summary (numerical variables):


,Churn,Tenure,CityTier,WarehouseToHome,HourSpendOnApp,NumberOfDeviceRegistered,SatisfactionScore,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
count,5630.000000,5366.000000,5630.000000,5379.000000,5375.000000,5630.000000,5630.000000,5630.000000,5630.000000,5365.000000,5374.000000,5372.000000,5323.000000,5630.000000
mean,0.168384,10.189899,1.654707,15.639896,2.931535,3.688988,3.066785,4.214032,0.284902,15.707922,1.751023,3.008004,4.543491,177.221492
std,0.374240,8.557241,0.915389,8.531475,0.721926,1.023999,1.380194,2.583586,0.451408,3.675485,1.894621,2.939680,3.654433,49.193869
min,0.000000,0.000000,1.000000,5.000000,0.000000,1.000000,1.000000,1.000000,0.000000,11.000000,0.000000,1.000000,0.000000,0.000000
25%,0.000000,2.000000,1.000000,9.000000,2.000000,3.000000,2.000000,2.000000,0.000000,13.000000,1.000000,1.000000,2.000000,146.000000
50%,0.000000,9.000000,1.000000,14.000000,3.000000,4.000000,3.000000,3.000000,0.000000,15.000000,1.000000,2.000000,3.000000,163.000000
75%,0.000000,16.000000,3.000000,20.000000,3.000000,4.000000,4.000000,6.000000,1.000000,18.000000,2.000000,3.000000,7.000000,196.000000
max,1.000000,61.000000,3.000000,127.000000,5.000000,6.000000,5.000000,22.000000,1.000000,26.000000,16.000000,16.000000,46.000000,325.000000



Categorical summary:


,PreferredLoginDevice,PreferredPaymentMode,Gender,PreferedOrderCat,MaritalStatus
count,5630,5630,5630,5630,5630
unique,3,7,2,6,3
top,Mobile Phone,Debit Card,Male,Laptop & Accessory,Married
freq,2765,2314,3384,2050,2986


## Nhận xét mô tả dữ liệu

### 1. Thời gian gắn bó của khách hàng (Tenure)
Thời gian gắn bó trung bình của khách hàng vào khoảng 10,19 tháng, với độ lệch chuẩn khá cao là 8,56. Điều này cho thấy mức độ phân tán lớn giữa các khách hàng về thời gian sử dụng dịch vụ. Sự khác biệt đáng kể này hàm ý rằng biến Tenure có thể đóng vai trò quan trọng trong việc ảnh hưởng đến khả năng rời bỏ và cần được xem xét kỹ hơn trong các bước mô hình hóa tiếp theo.

### 2. Khoảng cách từ kho đến nơi ở của khách hàng (Warehouse-to-Home Distance)
Khoảng cách lớn nhất từ kho đến nơi ở của khách hàng lên tới 127, trong khi giá trị trung vị chỉ vào khoảng 14. Sự chênh lệch lớn này cho thấy phân phối của biến có xu hướng lệch phải rõ rệt. Vì vậy, biến này có thể cần được biến đổi hoặc chuẩn hóa trước khi đưa vào huấn luyện mô hình nhằm hạn chế ảnh hưởng của các giá trị ngoại lai và giảm nguy cơ gây sai lệch kết quả.

### 3. Thời gian sử dụng ứng dụng (HourSpendOnApp)
Thời gian trung bình khách hàng sử dụng ứng dụng vào khoảng 2,93 giờ. Điều này phản ánh mức độ tương tác của người dùng với nền tảng nhìn chung tương đối ổn định trong toàn bộ tập khách hàng.

### 4. Các biến phân loại
Đối với các biến phân loại, điện thoại di động là thiết bị đăng nhập được sử dụng phổ biến nhất, thẻ ghi nợ là phương thức thanh toán chiếm ưu thế, và khách hàng đã kết hôn chiếm tỷ trọng lớn nhất trong mẫu nghiên cứu. Những đặc điểm này cung cấp thêm góc nhìn về hành vi sử dụng điển hình của khách hàng trên nền tảng và có thể góp phần giải thích sự khác biệt trong khả năng duy trì khách hàng.

In [8]:
TARGET_COL = "Churn"

DROP_COLS = ["CustomerID"]

CATEGORICAL_COLS = [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "Gender",
    "PreferedOrderCat",
    "MaritalStatus",
    "CityTier",
]

NUMERIC_COLS = [
    "WarehouseToHome",
    "HourSpendOnApp",
    "NumberOfDeviceRegistered",
    "SatisfactionScore",
    "NumberOfAddress",
    "Complain",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
    "CashbackAmount",
]

ALL_COLS = CATEGORICAL_COLS + NUMERIC_COLS + [TARGET_COL]
missing_cols = [c for c in ALL_COLS if c not in raw_data.columns]
assert len(missing_cols) == 0, f"Columns missing from dataset: {missing_cols}"
df = raw_data[ALL_COLS].copy()
print("Missing columns in dataset:", missing_cols)
df.head()


Missing columns in dataset: []


,PreferredLoginDevice,PreferredPaymentMode,Gender,PreferedOrderCat,MaritalStatus,CityTier,WarehouseToHome,HourSpendOnApp,NumberOfDeviceRegistered,SatisfactionScore,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,Churn
0,Mobile Phone,Debit Card,Female,Laptop & Accessory,Single,3,6.0,3.0,3,2,9,1,11.0,1.0,1.0,5.0,160,1
1,Phone,UPI,Male,Mobile,Single,1,8.0,3.0,4,3,7,1,15.0,0.0,1.0,0.0,121,1
2,Phone,Debit Card,Male,Mobile,Single,1,30.0,2.0,4,3,6,1,14.0,0.0,1.0,3.0,120,1
3,Phone,Debit Card,Male,Laptop & Accessory,Single,3,15.0,2.0,4,5,8,0,23.0,0.0,1.0,3.0,134,1
4,Phone,CC,Male,Mobile,Single,1,12.0,NaN,3,5,3,0,11.0,1.0,1.0,3.0,130,1


In [9]:
print("MISSING VALUES CHECK")

missing_count = raw_data.isna().sum()
missing_percent = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage (%)": missing_percent
})

display(missing_summary[missing_summary["Missing Count"] > 0]
        .sort_values(by="Missing Count", ascending=False))

print("\nTotal missing values in dataset:", df.isna().sum().sum())
print("Total missing percentage:",
      round((df.isna().sum().sum() / (df.shape[0] * df.shape[1])) * 100, 4), "%")

MISSING VALUES CHECK


,Missing Count,Missing Percentage (%)
DaySinceLastOrder,307,5.452931
OrderAmountHikeFromlastYear,265,4.706927
Tenure,264,4.689165
OrderCount,258,4.582593
CouponUsed,256,4.547069
HourSpendOnApp,255,4.529307
WarehouseToHome,251,4.458259



Total missing values in dataset: 1592
Total missing percentage: 1.5709 %


## Missing Data Analysis

Kết quả phân tích cho thấy tổng số giá trị thiếu trong bộ dữ liệu là **1.856**, chiếm khoảng **1,65%** tổng số dữ liệu. Mức độ thiếu này tương đối thấp và nhìn chung không có khả năng ảnh hưởng đáng kể đến chất lượng tổng thể của bộ dữ liệu.

Các biến có tỷ lệ giá trị thiếu cao nhất bao gồm:

- **DaySinceLastOrder** (5,45%)
- **OrderAmountHikeFromLastYear** (4,71%)
- **Tenure** (4,69%)
- **OrderCount** (4,58%)
- **CouponUsed** (4,55%)
- **HourSpendOnApp** (4,53%)
- **WarehouseToHome** (4,46%)

Có thể nhận thấy rằng các giá trị thiếu chủ yếu tập trung ở những biến phản ánh hành vi mua sắm và mức độ tương tác của khách hàng. Tuy nhiên, tỷ lệ thiếu của từng biến đều dưới 6%, cho thấy mức độ thiếu dữ liệu vẫn nằm trong ngưỡng chấp nhận được và có thể được xử lý bằng các kỹ thuật nội suy phù hợp mà không làm sai lệch đáng kể phân phối dữ liệu.

Với đặc điểm dữ liệu thiếu tương đối thấp và phân tán trên nhiều biến, nghiên cứu sẽ áp dụng các phương pháp xử lý giá trị thiếu dựa trên **tập huấn luyện** ở các bước tiếp theo. Cách tiếp cận này giúp đảm bảo tính toàn vẹn của dữ liệu, đồng thời hạn chế hiện tượng **rò rỉ dữ liệu (data leakage)** trong quá trình huấn luyện mô hình.

In [10]:
class_counts = df[TARGET_COL].value_counts()
class_ratio = df[TARGET_COL].value_counts(normalize=True)

print("\nClass Counts:")
display(class_counts)

print("\nClass Ratio:")
display(class_ratio)

print("\nChurn Percentage:",
      round(class_ratio[1] * 100, 2), "%")


Class Counts:


Churn
0    4682
1     948
Name: count, dtype: int64


Class Ratio:


Churn
0    0.831616
1    0.168384
Name: proportion, dtype: float64


Churn Percentage: 16.84 %


## Class Imbalance Analysis

Kết quả thống kê cho thấy trong tổng số **5.630 quan sát**, có **4.682 khách hàng không rời bỏ** (chiếm **83,16%**) và **948 khách hàng rời bỏ** (chiếm **16,84%**).

Điều này cho thấy bộ dữ liệu tồn tại **sự mất cân bằng lớp ở mức vừa phải**, trong đó nhóm khách hàng không rời bỏ chiếm tỷ trọng lớn hơn đáng kể so với nhóm khách hàng rời bỏ.

Mặc dù mức độ mất cân bằng này chưa phải quá nghiêm trọng, nhưng nếu mô hình chỉ được đánh giá dựa trên **độ chính xác (accuracy)** thì kết quả có thể bị thiên lệch về lớp chiếm đa số.

Vì vậy, nghiên cứu này ưu tiên sử dụng các chỉ số đánh giá như **F1-score**, **Recall**, và **ROC-AUC** nhằm phản ánh tốt hơn khả năng của mô hình trong việc nhận diện đúng những khách hàng có nguy cơ rời bỏ.

#### **3. Initial Data Cleaning**           

In [11]:
dup_count = df.duplicated().sum()
print("Duplicate rows before:", dup_count)
if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)

Duplicate rows before: 563
Shape after dropping duplicates: (5067, 18)


***
Trước khi xây dựng mô hình, bộ dữ liệu được kiểm tra hiện tượng trùng lặp quan sát. Kết quả cho thấy có **563 dòng trùng lặp hoàn toàn**. Các dòng này được loại bỏ nhằm tránh việc một số hồ sơ khách hàng xuất hiện lặp lại nhiều lần, từ đó làm sai lệch phân phối dữ liệu và ảnh hưởng đến kết quả huấn luyện mô hình. Sau khi loại bỏ các quan sát trùng lặp, bộ dữ liệu còn lại **5.067 quan sát và 18 biến**.
***

In [ ]:
print("CHECKING PROFILE OVERLAP BETWEEN SPLITS")

profile_cols = list(X_train_final.columns)

train_profiles = set(X_train_final[profile_cols].astype(str).agg("|".join, axis=1))
val_profiles = set(X_val_final[profile_cols].astype(str).agg("|".join, axis=1))
test_profiles = set(X_test_final[profile_cols].astype(str).agg("|".join, axis=1))

overlap_train_val = train_profiles & val_profiles
overlap_train_test = train_profiles & test_profiles
overlap_val_test = val_profiles & test_profiles

print("Overlap train-val:", len(overlap_train_val))
print("Overlap train-test:", len(overlap_train_test))
print("Overlap val-test:", len(overlap_val_test))

***
Sau khi loại bỏ các quan sát trùng lặp hoàn toàn, nghiên cứu tiếp tục kiểm tra mức độ trùng lặp hồ sơ đặc trưng giữa các tập train, validation và test. Kết quả cho thấy **không có sự trùng lặp profile** giữa **train và validation**, cũng như giữa **validation và test**. Chỉ ghi nhận **1 profile đặc trưng trùng nhau giữa train và test**.

Kết quả này cho thấy quá trình chia dữ liệu nhìn chung được thực hiện tương đối sạch, không xuất hiện hiện tượng chồng lấn đáng kể giữa các tập dữ liệu. Trường hợp trùng lặp duy nhất giữa train và test có thể xuất phát từ việc hai quan sát có cùng giá trị trên toàn bộ các biến đầu vào, dù không nhất thiết là cùng một bản ghi hay cùng nhãn mục tiêu. Với số lượng rất nhỏ, mức độ ảnh hưởng đến kết quả mô hình được xem là không đáng kể.
***

In [ ]:
for col in CATEGORICAL_COLS:
    if col in df.columns:
        # keep NaN as NaN, only strip non-null values
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

print("Trimmed whitespace in categorical columns.")


In [ ]:
print("\n--- Basic validity checks ---")

# Target values
print("Target unique values:", sorted(df[TARGET_COL].dropna().unique().tolist()))

# CityTier should be small set (typically 1/2/3)
if "CityTier" in df.columns:
    print("\nCityTier value counts:")
    display(df["CityTier"].value_counts(dropna=False))

# SatisfactionScore often 1-5 (depending on dataset)
if "SatisfactionScore" in df.columns:
    print("\nSatisfactionScore value counts:")
    display(df["SatisfactionScore"].value_counts(dropna=False).sort_index())

print("\n========== CLEANING COMPLETED ==========")
df.head()

In [ ]:
replacements = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone",
        "Mobile": "Mobile Phone",
        "MobilePhone": "Mobile Phone",
        "Laptop": "Computer",
    },
    "PreferredPaymentMode": {
        "CC": "Credit Card",
        "CreditCard": "Credit Card",
        "DebitCard": "Debit Card",
        "COD": "Cash on Delivery",
        "CashonDelivery": "Cash on Delivery",
        "E wallet": "E-wallet",
        "EWallet": "E-wallet",
    },
}

for col, mp in replacements.items():
    if col in df.columns:
        df[col] = df[col].replace(mp)

# Strip whitespace safely
for col in CATEGORICAL_COLS:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

print("Category normalization completed.")
df.head()

#### **4. Experimental Data Split**

In [ ]:

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

# 1) Split off TEST (15%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=SEED
)

In [ ]:
# val proportion in remaining = 0.15 / 0.85 = 0.1764706
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.1764706,
    stratify=y_train_val,
    random_state=SEED
)

print("\nShapes:")
print("Train:", X_train.shape, "| y_train:", y_train.shape)
print("Val  :", X_val.shape,   "| y_val  :", y_val.shape)
print("Test :", X_test.shape,  "| y_test :", y_test.shape)


In [ ]:
def show_ratio(name, y_series):
    ratio = y_series.value_counts(normalize=True)
    print(f"\n{name} churn ratio:")
    print(ratio)
    if 1 in ratio.index:
        print(f"{name} churn percentage: {ratio.loc[1]*100:.2f}%")

show_ratio("Train", y_train)
show_ratio("Validation", y_val)
show_ratio("Test", y_test)

#### **5.Handling Missing Data**

In [ ]:
for c in CATEGORICAL_COLS + NUMERIC_COLS:
    if c not in X_train.columns:
        raise ValueError(f"Column {c} not found in X_train. Check your schema or split step.")

In [ ]:
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

In [ ]:
X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train[NUMERIC_COLS]),
    columns=NUMERIC_COLS, index=X_train.index
)

X_train_cat = pd.DataFrame(
    cat_imputer.fit_transform(X_train[CATEGORICAL_COLS]),
    columns=CATEGORICAL_COLS, index=X_train.index
).astype("string")

In [ ]:
X_val_num = pd.DataFrame(
    num_imputer.transform(X_val[NUMERIC_COLS]),
    columns=NUMERIC_COLS, index=X_val.index
)
X_val_cat = pd.DataFrame(
    cat_imputer.transform(X_val[CATEGORICAL_COLS]),
    columns=CATEGORICAL_COLS, index=X_val.index
).astype("string")

X_test_num = pd.DataFrame(
    num_imputer.transform(X_test[NUMERIC_COLS]),
    columns=NUMERIC_COLS, index=X_test.index
)
X_test_cat = pd.DataFrame(
    cat_imputer.transform(X_test[CATEGORICAL_COLS]),
    columns=CATEGORICAL_COLS, index=X_test.index
).astype("string")

In [ ]:
X_train_imp = pd.concat([X_train_cat, X_train_num], axis=1)[CATEGORICAL_COLS + NUMERIC_COLS]
X_val_imp   = pd.concat([X_val_cat, X_val_num], axis=1)[CATEGORICAL_COLS + NUMERIC_COLS]
X_test_imp  = pd.concat([X_test_cat, X_test_num], axis=1)[CATEGORICAL_COLS + NUMERIC_COLS]

print("Shapes after imputation:")
print("X_train_imp:", X_train_imp.shape)
print("X_val_imp  :", X_val_imp.shape)
print("X_test_imp :", X_test_imp.shape)

print("\nMissing values after imputation (should be 0):")
print("Train missing:", int(X_train_imp.isna().sum().sum()))
print("Val missing  :", int(X_val_imp.isna().sum().sum()))
print("Test missing :", int(X_test_imp.isna().sum().sum()))

X_train_imp.head()

#### **6. Encoding and Scaling**

In [ ]:
label_encoders = {}
X_train_cat = X_train_imp[CATEGORICAL_COLS].copy()
X_val_cat   = X_val_imp[CATEGORICAL_COLS].copy()
X_test_cat  = X_test_imp[CATEGORICAL_COLS].copy()

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    # fit on TRAIN only
    X_train_cat[col] = le.fit_transform(X_train_cat[col].astype(str))
    # transform VAL/TEST
    X_val_cat[col]   = le.transform(X_val_cat[col].astype(str))
    X_test_cat[col]  = le.transform(X_test_cat[col].astype(str))
    label_encoders[col] = le

print("Label encoding completed for categorical features.")


In [ ]:
scaler = StandardScaler()

X_train_num = pd.DataFrame(
    scaler.fit_transform(X_train_imp[NUMERIC_COLS]),
    columns=NUMERIC_COLS,
    index=X_train_imp.index
)

X_val_num = pd.DataFrame(
    scaler.transform(X_val_imp[NUMERIC_COLS]),
    columns=NUMERIC_COLS,
    index=X_val_imp.index
)

X_test_num = pd.DataFrame(
    scaler.transform(X_test_imp[NUMERIC_COLS]),
    columns=NUMERIC_COLS,
    index=X_test_imp.index
)

print("Scaling completed for numerical features.")

In [ ]:
FINAL_COLS = CATEGORICAL_COLS + NUMERIC_COLS

X_train_final = pd.concat([X_train_cat, X_train_num], axis=1)[FINAL_COLS]
X_val_final   = pd.concat([X_val_cat,   X_val_num],   axis=1)[FINAL_COLS]
X_test_final  = pd.concat([X_test_cat,  X_test_num],  axis=1)[FINAL_COLS]

print("\nFinal shapes:")
print("X_train_final:", X_train_final.shape)
print("X_val_final  :", X_val_final.shape)
print("X_test_final :", X_test_final.shape)

In [ ]:
# Prepare TabNet metadata
# - cat_idxs: indices of categorical columns
# - cat_dims: number of unique categories in each col (based on TRAIN)
# -----------------------------
cat_idxs = list(range(len(CATEGORICAL_COLS)))
cat_dims = [int(X_train_cat[c].nunique()) for c in CATEGORICAL_COLS]

print("\nTabNet categorical indices (cat_idxs):", cat_idxs)
print("TabNet categorical cardinalities (cat_dims):", cat_dims)


In [ ]:
X_train_np = X_train_final.values.astype(np.float32)
X_val_np   = X_val_final.values.astype(np.float32)
X_test_np  = X_test_final.values.astype(np.float32)

y_train_np = y_train.values.astype(int)
y_val_np   = y_val.values.astype(int)
y_test_np  = y_test.values.astype(int)

print("\nNumpy arrays ready for TabNet:")
print("X_train_np:", X_train_np.shape, "| y_train_np:", y_train_np.shape)
print("X_val_np  :", X_val_np.shape,   "| y_val_np  :", y_val_np.shape)
print("X_test_np :", X_test_np.shape,  "| y_test_np :", y_test_np.shape)


In [ ]:
X_train_final.head()

#### **7. Summary of Prepared Data**

In [ ]:
print("=" * 60)
print("SUMMARY OF PREPARED DATA")
print("=" * 60)

print("\n[1] Final Feature List")
print(FINAL_COLS)

print("\n[2] Final Matrix Shapes")
print("X_train_final:", X_train_final.shape)
print("X_val_final:", X_val_final.shape)
print("X_test_final:", X_test_final.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

print("\n[3] Missing Value Check")
print("Train:", int(np.isnan(X_train_np).sum()))
print("Val  :", int(np.isnan(X_val_np).sum()))
print("Test :", int(np.isnan(X_test_np).sum()))

print("\n[4] Class Ratio Check")
print("Train:", y_train.value_counts(normalize=True).to_dict())
print("Val  :", y_val.value_counts(normalize=True).to_dict())
print("Test :", y_test.value_counts(normalize=True).to_dict())

print("\n[5] Feature Group Summary")
print("Number of final features:", len(FINAL_COLS))
print("Number of categorical features:", len(cat_idxs))
print("Number of numerical features:", len(FINAL_COLS) - len(cat_idxs))

print("\n[6] TabNet Categorical Settings")
print("cat_idxs:", cat_idxs)
print("cat_dims:", cat_dims)

#### **8. Export Artifacts**

In [ ]:
os.makedirs("artifacts/data", exist_ok=True)
os.makedirs("artifacts/results", exist_ok=True)
os.makedirs("artifacts/models", exist_ok=True)
os.makedirs("artifacts/figures", exist_ok=True)

# save tabular data
X_train_final.to_csv("artifacts/data/X_train_final.csv", index=False)
X_val_final.to_csv("artifacts/data/X_val_final.csv", index=False)
X_test_final.to_csv("artifacts/data/X_test_final.csv", index=False)

X_train.to_csv("artifacts/data/X_train_raw.csv", index=False)
X_val.to_csv("artifacts/data/X_val_raw.csv", index=False)
X_test.to_csv("artifacts/data/X_test_raw.csv", index=False)

pd.Series(y_train).to_csv("artifacts/data/y_train.csv", index=False)
pd.Series(y_val).to_csv("artifacts/data/y_val.csv", index=False)
pd.Series(y_test).to_csv("artifacts/data/y_test.csv", index=False)

# numpy for tabnet
np.save("artifacts/data/X_train_np.npy", X_train_np)
np.save("artifacts/data/X_val_np.npy", X_val_np)
np.save("artifacts/data/X_test_np.npy", X_test_np)

np.save("artifacts/data/y_train_np.npy", y_train_np)
np.save("artifacts/data/y_val_np.npy", y_val_np)
np.save("artifacts/data/y_test_np.npy", y_test_np)

# Save cleaned full dataset (before final matrix construction) for EDA
df.to_csv("artifacts/data/df_clean.csv", index=False)

meta = {
    "FINAL_COLS": FINAL_COLS,
    "cat_idxs": cat_idxs,
    "cat_dims": cat_dims,
    "TARGET_COL": TARGET_COL,
    "CATEGORICAL_COLS": CATEGORICAL_COLS,
    "NUMERIC_COLS": NUMERIC_COLS
}

with open("artifacts/data/meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved all preprocessing artifacts.")

This notebook establishes the single source of truth for all downstream analyses. All preprocessing steps are fitted on the training split only. The resulting processed artifacts are exported and reused consistently in the EDA, modeling, and evaluation notebooks.